# 07 - True First PC Service

This notebook checks whether researchers in `pc_members.parquet` are truly
first time PC members, or only first observed in the 2017-2025 PC data.

I use two source families:

- Researchr profile histories, which list older committee service when the
  profile page contains it.
- ACM front matter PDFs, where committee sections can reveal older PC, EPC,
  chair, steering, or organizing roles.

The output keeps both narrow PC evidence and broader review service evidence.
This matters because "no prior PC" and "no prior PC/EPC/Chair" are different
filters.


## 1 - Setup

In [1]:
import re
import time
import unicodedata
import sys
from pathlib import Path

import pandas as pd
import pypdf
import requests
from bs4 import BeautifulSoup


In [2]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_1_data_dir = project_folder / "step_1_data"
step_1_artifacts_dir = project_folder / "step_1_artifacts"
prepared_dir = step_1_data_dir / "prepared"
intermediate_dir = step_1_data_dir / "intermediate"
raw_dir = step_1_data_dir / "raw"
summary_tables_dir = step_1_artifacts_dir / "summary_tables"
dependency_tables_dir = step_1_artifacts_dir / "dependency_tables"
check_tables_dir = step_1_artifacts_dir / "check_tables"
profile_html_dir = raw_dir / "researchr_profile_htmls"
pdf_dir = project_folder / "assets" / "validation_pdfs"

intermediate_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)
profile_html_dir.mkdir(parents=True, exist_ok=True)

TARGET_CONFERENCES = ["ICFP", "OOPSLA", "PLDI", "POPL"]
REFETCH_PROFILE_HTML = not use_existing_data
FETCH_MISSING_PROFILES = allow_network and not use_existing_data
REQUEST_SLEEP_SECONDS = 0.25

print(project_folder)
print(profile_html_dir)
print(f"Run mode: {run_mode}")


/Users/endersari/2026-02-citations-vs-pc-memberships
/Users/endersari/2026-02-citations-vs-pc-memberships/step_1_data/raw/researchr_profile_htmls
Run mode: fast


## 2 - Load Current PC Members

In [3]:
pc_members = pd.read_parquet(prepared_dir / "pc_members.parquet")

required = {
    "conference", "year", "name", "researcher_id",
    "researchr_id", "canonical_researchr_id", "pc_it",
}
missing = sorted(required - set(pc_members.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

pc_members = pc_members.loc[pc_members["pc_it"] == True].copy()
pc_members["conference"] = pc_members["conference"].astype(str).str.upper()
pc_members["year"] = pc_members["year"].astype(int)
pc_members["researcher_id"] = pc_members["canonical_researchr_id"].astype(str)

pc_events = (
    pc_members[["researcher_id", "conference", "year"]]
    .drop_duplicates()
    .sort_values(["researcher_id", "conference", "year"])
    .reset_index(drop=True)
)

people = (
    pc_members
    .sort_values(["researcher_id", "year", "conference"])
    .drop_duplicates("researcher_id")
    [["researcher_id", "name", "person_url"]]
    .reset_index(drop=True)
)

first_observed_conf = (
    pc_events
    .groupby(["researcher_id", "conference"], as_index=False)
    .agg(first_observed_pc_year_conf=("year", "min"))
)

first_observed_any = (
    pc_events
    .groupby("researcher_id", as_index=False)
    .agg(first_observed_pc_year_any_target=("year", "min"))
)

print("pc rows:", pc_members.shape)
print("unique researchers:", people["researcher_id"].nunique())
print("researcher-conference pairs:", len(first_observed_conf))
display(first_observed_conf.head())


pc rows: (2180, 18)
unique researchers: 952
researcher-conference pairs: 1496


,researcher_id,conference,first_observed_pc_year_conf
0,aaronbembenek,PLDI,2025
1,aaronstump,POPL,2019
2,abhinavverma1,PLDI,2023
3,adamchlipala,ICFP,2017
4,adamchlipala,PLDI,2018


## 3 - Name Normalization

In [4]:
fold_map = str.maketrans({
    "ø": "o", "Ø": "O",
    "æ": "ae", "Æ": "AE",
    "œ": "oe", "Œ": "OE",
    "ß": "ss",
    "ł": "l", "Ł": "L",
    "đ": "d", "Đ": "D",
    "þ": "th", "Þ": "Th",
    "ð": "d", "Ð": "D",
    "’": "'", "‘": "'",
    "“": '"', "”": '"',
})


def fold_text(value):
    if not isinstance(value, str):
        return ""
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.translate(fold_map)
    value = re.sub(r"\([^)]*\)", " ", value)
    value = value.replace(".", " ").replace(",", " ")
    return " ".join(value.lower().split())


def normalize_name(name):
    folded = fold_text(name)
    folded = re.sub(r"[^a-z0-9 ]+", " ", folded)
    return re.sub(r"\s+", " ", folded).strip()


name_map_path = dependency_tables_dir / "name_map_used_for_source_comparison.csv"
if name_map_path.exists():
    name_map_table = pd.read_csv(name_map_path)
    NAME_MAP = dict(zip(name_map_table["name_variant"], name_map_table["name_canonical"]))
else:
    name_map_table = pd.DataFrame(columns=["name_variant", "name_canonical"])
    NAME_MAP = {}

name_map_norm = {normalize_name(k): v for k, v in NAME_MAP.items()}
reverse_name_map = {}
for variant, canonical in NAME_MAP.items():
    reverse_name_map.setdefault(normalize_name(canonical), set()).add(variant)


def lookup_name(name):
    if not isinstance(name, str) or not name:
        return name
    if name in NAME_MAP:
        return NAME_MAP[name]
    return name_map_norm.get(normalize_name(name), name)


names_by_researcher = (
    pc_members
    .groupby("researcher_id")["name"]
    .apply(lambda s: sorted(set(x for x in s if isinstance(x, str) and x)))
    .to_dict()
)


def candidate_names_for_researcher(researcher_id):
    names = set(names_by_researcher.get(researcher_id, []))
    for name in list(names):
        canonical = lookup_name(name)
        names.add(canonical)
        names.update(reverse_name_map.get(normalize_name(canonical), set()))
    return sorted(n for n in names if isinstance(n, str) and n)


print("name-map entries:", len(NAME_MAP))
print("example candidate names:")
for rid in people["researcher_id"].head(3):
    print(rid, "->", candidate_names_for_researcher(rid))


name-map entries: 128
example candidate names:
aaronbembenek -> ['Aaron Bembenek']
aaronstump -> ['Aaron Stump']
abhinavverma1 -> ['Abhinav Verma']


## 4 - Researchr Profile Histories

In [5]:
PROFILE_URL = "https://conf.researchr.org/profile/conf/{researcher_id}"


def safe_profile_filename(researcher_id):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(researcher_id)) + ".html"



def get_profile_html(researcher_id):
    url = PROFILE_URL.format(researcher_id=researcher_id)
    cache_path = profile_html_dir / safe_profile_filename(researcher_id)

    if cache_path.exists() and not REFETCH_PROFILE_HTML:
        return cache_path.read_text(encoding="utf-8", errors="replace"), "cached", url, str(cache_path)

    if not FETCH_MISSING_PROFILES:
        return None, "missing_not_fetched", url, ""

    try:
        response = requests.get(
            url,
            timeout=25,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        status = f"http_{response.status_code}"
        if response.status_code == 200 and response.text:
            cache_path.write_text(response.text, encoding="utf-8")
            time.sleep(REQUEST_SLEEP_SECONDS)
            return response.text, "fetched", url, str(cache_path)
        return None, status, url, ""
    except Exception as exc:
        return None, f"fetch_error: {type(exc).__name__}", url, ""


In [6]:
SUB_EVENT_KEYWORDS = [
    "student research competition",
    "doctoral symposium",
    "workshop",
    "poster",
    "demonstration",
    "tutorial",
    "panel",
]


def classify_sub_event(track, committee):
    haystack = f"{track} {committee}".lower()
    if "student research competition" in haystack or " src " in f" {haystack} ":
        return "SRC"
    if "doctoral symposium" in haystack:
        return "Doctoral Symposium"
    if "workshop" in haystack:
        return "Workshop"
    if "panel" in haystack:
        return "Panel"
    if "poster" in haystack and "committee" in haystack:
        return "Posters"
    if ("demonstration" in haystack or "tutorial" in haystack) and "committee" in haystack:
        return "Tutorial/Demo"
    return None


def normalize_profile_role(raw_role):
    if not raw_role:
        return ""

    match = re.match(
        r"^(.+?)\s+in\s+(.+?)(?:\s+within the\s+(.+?)-track)?\s*$",
        raw_role,
        re.I,
    )
    if match:
        qualifier = match.group(1).strip()
        committee = match.group(2).strip()
        track = (match.group(3) or "").strip()
    else:
        qualifier = raw_role.strip()
        committee = ""
        track = ""

    qual_lc = qualifier.lower()
    comm_lc = committee.lower()
    track_lc = track.lower()
    is_chair = "chair" in qual_lc

    sub_event = classify_sub_event(track, committee)
    if sub_event:
        return f"{sub_event} Chair" if is_chair else f"{sub_event} Member"

    if "artifact" in track_lc or "artifact" in comm_lc:
        return "AEC Chair" if is_chair else "AEC Member"

    if "general chair" in qual_lc:
        return "General Chair"
    if "workshops chair" in qual_lc or "workshop chair" in qual_lc:
        return "Workshops Chair"
    if "publicity chair" in qual_lc:
        return "Publicity Chair"
    if "publications chair" in qual_lc:
        return "Publications Chair"
    if "session chair" in qual_lc:
        return "Session Chair"

    if "external review" in comm_lc or "extended review" in comm_lc:
        return "EPC Chair" if is_chair else "EPC Member"
    if "steering committee" in comm_lc:
        return "Steering Chair" if is_chair else "Steering Committee"
    if "organizing" in comm_lc or "organising" in comm_lc:
        return "Organizing"
    if (
        "program committee" in comm_lc
        or "review committee" in comm_lc
        or "research papers" in comm_lc
        or "papers and events" in comm_lc
    ):
        return "PC Chair" if is_chair else "PC Member"
    if re.fullmatch(r"[A-Za-z]{2,8}", committee):
        return "PC Chair" if is_chair else "PC Member"

    return raw_role.strip()


def extract_conference_from_profile(track, link_text):
    track_upper = (track or "").strip().upper()
    if track_upper in TARGET_CONFERENCES:
        return track_upper
    text_upper = f"{track or ''} {link_text or ''}".upper()
    for conf in TARGET_CONFERENCES:
        if re.search(rf"\b{conf}\b", text_upper):
            return conf
    return track_upper


def parse_researchr_profile(html, researcher_id):
    profile = {
        "researcher_id": researcher_id,
        "display_name": None,
        "affiliation": None,
        "country": None,
    }
    services = []

    if not html:
        return profile, services

    soup = BeautifulSoup(html, "html.parser")

    title = soup.find("title")
    if title:
        title_text = title.get_text(strip=True)
        title_text = re.sub(r"\s*-\s*$", "", title_text)
        title_text = re.sub(r"\s*-\s*[^-]*$", "", title_text) if " - " in title_text else title_text
        profile["display_name"] = title_text or None

    for item in soup.find_all("div", class_="profile-item"):
        heading = item.find("span", class_="profile-item-heading")
        if not heading:
            continue
        label = heading.get_text(strip=True)
        full = item.get_text(" ", strip=True)
        value = full[len(label):].strip() if full.startswith(label) else full
        if label == "Name:" and not profile["display_name"]:
            profile["display_name"] = value or None
        elif label == "Affiliation:":
            profile["affiliation"] = value or None
        elif label == "Country:":
            profile["country"] = value or None

    for year_block in soup.find_all("div", class_="contribution-year"):
        h3 = year_block.find("h3")
        if not h3:
            continue
        try:
            year = int(h3.get_text(strip=True))
        except Exception:
            continue

        for sub in year_block.find_all("div", recursive=False):
            h4 = sub.find("h4")
            if not h4:
                continue
            track = h4.get_text(strip=True)
            for li in sub.select("ul.block > li"):
                small = li.find("small")
                if not small:
                    continue
                kind = (small.get("title") or "").strip().lower()
                if kind != "member of committee":
                    continue
                link = li.find("a")
                if not link:
                    continue
                raw_role = link.get_text(" ", strip=True)
                match = re.match(
                    r"^(.+?)\s+in\s+(.+?)(?:\s+within the\s+(.+?)-track)?\s*$",
                    raw_role,
                )
                raw_track = (match.group(3) if match and match.group(3) else track)
                services.append({
                    "researcher_id": researcher_id,
                    "conference": extract_conference_from_profile(track, raw_role),
                    "year": year,
                    "role": normalize_profile_role(raw_role),
                    "raw_role": raw_role,
                    "raw_track": raw_track,
                    "source": "Researchr profile",
                })

    return profile, services


In [7]:
base = project_folder

log = pd.read_parquet(base / "step_1_data/intermediate/researchr_profile_fetch_log.parquet")
services = pd.read_parquet(base / "step_1_data/intermediate/researchr_profile_services.parquet")

print(log["profile_fetch_status"].value_counts(dropna=False))
print(services.shape)

print("fetch log:", log.shape)
print(log["profile_fetch_status"].value_counts(dropna=False))
print("services:", services.shape)
if len(services):
    print("target services:", int(services["is_target_conference"].sum()))


profile_fetch_status
cached    952
Name: count, dtype: int64
(15214, 10)
fetch log: (952, 9)
profile_fetch_status
cached    952
Name: count, dtype: int64
services: (15214, 10)
target services: 5292


In [8]:
profile_log_path = intermediate_dir / "researchr_profile_fetch_log.parquet"
profile_services_path = intermediate_dir / "researchr_profile_services.parquet"

use_profile_cache = (
    use_existing_data
    and profile_log_path.exists()
    and profile_services_path.exists()
)

if use_profile_cache:
    profiles = pd.read_parquet(profile_log_path)
    profile_services = pd.read_parquet(profile_services_path)
    all_missing = (
        "profile_fetch_status" in profiles.columns
        and profiles["profile_fetch_status"].eq("missing_not_fetched").all()
    )
    if len(profile_services) == 0 and all_missing:
        use_profile_cache = False

if use_profile_cache:
    print("Loaded cached Researchr profile histories")
else:
    profile_rows = []
    service_rows = []

    for i, row in people.iterrows():
        researcher_id = row["researcher_id"]
        html, status, url, cache_path = get_profile_html(researcher_id)
        profile, services = parse_researchr_profile(html, researcher_id)
        profile_rows.append({
            **profile,
            "name_in_pc_members": row["name"],
            "profile_url": url,
            "profile_fetch_status": status,
            "profile_cache_path": cache_path,
            "n_profile_services": len(services),
        })
        service_rows.extend(services)

        if (i + 1) % 100 == 0 or i + 1 == len(people):
            print(f"{i + 1}/{len(people)} profiles processed")

    profiles = pd.DataFrame(profile_rows)
    profile_services = pd.DataFrame(service_rows)

    if len(profile_services) == 0 and not FETCH_MISSING_PROFILES:
        raise RuntimeError(
            "Researchr profile histories are missing from the cache. "
            "Restore step_1_data/intermediate/researchr_profile_services.parquet "
            "or enable network fetching."
        )

if len(profile_services):
    profile_services["conference"] = profile_services["conference"].astype(str).str.upper()
    profile_services["is_target_conference"] = profile_services["conference"].isin(TARGET_CONFERENCES)
    profile_services["is_profile_main_pc"] = profile_services["role"].isin({"PC Member", "PC Chair"})
    profile_services["is_profile_broad_service"] = (
        profile_services["role"].isin({
            "PC Member", "PC Chair",
            "EPC Member", "EPC Chair",
            "General Chair", "Steering Chair", "Steering Committee",
            "Organizing", "Workshops Chair", "Publicity Chair", "Publications Chair",
        })
    )
else:
    profile_services = pd.DataFrame(columns=[
        "researcher_id", "conference", "year", "role", "raw_role", "raw_track",
        "source", "is_target_conference", "is_profile_main_pc", "is_profile_broad_service",
    ])

print("profiles:", profiles.shape)
print(profiles["profile_fetch_status"].value_counts(dropna=False).to_string())
print("profile services:", profile_services.shape)
if len(profile_services):
    display(profile_services.query("is_target_conference").head())


Loaded cached Researchr profile histories
profiles: (952, 9)
profile_fetch_status
cached    952
profile services: (15214, 10)


,researcher_id,conference,year,role,raw_role,raw_track,source,is_target_conference,is_profile_main_pc,is_profile_broad_service
1,aaronbembenek,OOPSLA,2026,PC Member,Committee Member in OOPSLA Review Committee wi...,OOPSLA,Researchr profile,True,True,True
4,aaronbembenek,PLDI,2025,PC Member,Committee Member in PLDI Review Committee with...,PLDI Research Papers,Researchr profile,True,True,True
7,aaronstump,POPL,2023,PC Member,Committee Member in Program Committee within t...,POPL,Researchr profile,True,True,True
10,aaronstump,ICFP,2020,EPC Member,Committee Member in External Review Committee ...,ICFP Program,Researchr profile,True,False,True
11,aaronstump,POPL,2019,PC Member,Committee Member in Program Committee within t...,Research Papers,Researchr profile,True,True,True


## 5 - ACM Front Matter PDF Scan

In [9]:
def read_pdf_text(pdf_path):
    try:
        reader = pypdf.PdfReader(str(pdf_path))
        return "\n".join((page.extract_text() or "") for page in reader.pages)
    except Exception as exc:
        print(f"PDF read failed: {pdf_path} ({type(exc).__name__})")
        return ""


def pdf_text_normalized(pdf_path):
    return " " + fold_text(read_pdf_text(pdf_path)) + " "


def name_in_pdf(name, pdf_norm_text, gap=50):
    tokens = fold_text(name).split()
    if not tokens:
        return False
    first = re.escape(tokens[0])
    if len(tokens) == 1:
        return bool(re.search(rf"\s{first}\s", pdf_norm_text))
    last = re.escape(tokens[-1])
    return bool(re.search(rf"\s{first}\b.{{0,{gap}}}\b{last}\s", pdf_norm_text, re.S))


SECTION_MARKERS = [
    ("research program committee", "PC"),
    ("primary review committee", "PC"),
    ("program committee", "PC"),
    ("review committee", "PC"),
    ("secondary review committee", "EPC"),
    ("extended review committee", "EPC"),
    ("additional review committee", "EPC"),
    ("external review committee", "EPC"),
    ("additional ad hoc reviews", "EPC"),
    ("additional reviewers", "EPC"),
    ("additional oopsla reviewers", "EPC"),
    ("additional icfp reviewers", "EPC"),
    ("additional popl reviewers", "EPC"),
    ("additional pldi reviewers", "EPC"),
    ("external reviewers", "EPC"),
    ("sub-reviewers", "EPC"),
    ("sub reviewers", "EPC"),
    ("research program chair", "Chair"),
    ("general chair", "Chair"),
    ("program chair", "Chair"),
    ("workshops chair", "Chair"),
    ("publications chair", "Chair"),
    ("publicity chair", "Chair"),
    ("local arrangements", "Chair"),
    ("doctoral symposium", "Chair"),
    ("steering committee", "Steering"),
    ("organizing committee", "Organizing"),
    ("organising committee", "Organizing"),
]

NON_SERVICE_MARKERS = [
    "table of contents",
    "author index",
    "contents",
    "session 1", "session 2", "session 3", "session 4", "session 5",
    "session 6", "session 7", "session 8", "session 9",
    "invited talk",
    "keynote",
    "frontmatter",
    "back matter",
    "sponsors",
    "supporters",
    "additional onward! reviewers",
    "additional onward reviewers",
    "onward! program committee",
    "onward program committee",
    "onward! review committee",
]

ROLE_PRIORITY = {
    "Other": 0,
    "Organizing": 1,
    "Steering": 2,
    "Chair": 3,
    "EPC": 4,
    "PC": 5,
}


def classify_match_position(pos, pdf_norm_text, lookback=5000):
    preceding = pdf_norm_text[max(0, pos - lookback):pos]
    found = []
    for phrase, role in SECTION_MARKERS:
        for match in re.finditer(re.escape(phrase), preceding):
            found.append((match.start(), match.end(), role, len(phrase)))
    for phrase in NON_SERVICE_MARKERS:
        for match in re.finditer(re.escape(phrase), preceding):
            found.append((match.start(), match.end(), "Other", len(phrase)))
    if not found:
        return "Other"

    found.sort(key=lambda x: -x[3])
    kept = []
    for start, end, role, length in found:
        subsumed = any(ks <= start and end <= ke for ks, ke, _role, _length in kept)
        if not subsumed:
            kept.append((start, end, role, length))
    kept.sort(key=lambda x: x[0])
    return kept[-1][2]


def name_in_pdf_with_role(name, pdf_norm_text, gap=50, lookback=5000):
    tokens = fold_text(name).split()
    if not tokens:
        return None
    first = re.escape(tokens[0])
    if len(tokens) == 1:
        pattern = rf"\s{first}\s"
    else:
        last = re.escape(tokens[-1])
        pattern = rf"\s{first}\b.{{0,{gap}}}\b{last}\s"

    best_role = None
    best_priority = -1
    for match in re.finditer(pattern, pdf_norm_text, re.S):
        role = classify_match_position(match.start(), pdf_norm_text, lookback=lookback)
        priority = ROLE_PRIORITY.get(role, 0)
        if priority > best_priority:
            best_priority = priority
            best_role = role
    return best_role


In [10]:
def pdf_year_from_name(conf, filename):
    conf_lc = conf.lower()
    match = re.match(rf"^{conf_lc}_(\d{{4}})(?:_oopsla[12])?_acm_frontmatter_", filename)
    if match:
        return int(match.group(1))
    return None


pdf_text_by_cell = {}
pdf_paths_by_cell = {}

for conf in TARGET_CONFERENCES:
    folder = pdf_dir / conf.lower()
    by_year = {}
    for path in sorted(folder.glob("*.pdf")):
        year = pdf_year_from_name(conf, path.name)
        if year is None:
            continue
        by_year.setdefault(year, []).append(path)

    for year, paths in sorted(by_year.items()):
        pdf_text_by_cell[(conf, year)] = "  ".join(pdf_text_normalized(path) for path in paths)
        pdf_paths_by_cell[(conf, year)] = [str(path.relative_to(repo)) for path in paths]

print("PDF cells:", len(pdf_text_by_cell))
for conf in TARGET_CONFERENCES:
    years = sorted(year for (c, year) in pdf_text_by_cell if c == conf)
    print(conf, len(years), f"{min(years)}-{max(years)}" if years else "missing")


PDF cells: 85
ICFP 21 2005-2025
OOPSLA 22 2005-2026
PLDI 21 2005-2025
POPL 21 2005-2025


In [11]:
pdf_rows = []

for i, person in people.iterrows():
    researcher_id = person["researcher_id"]
    candidates = candidate_names_for_researcher(researcher_id)
    for conf in TARGET_CONFERENCES:
        for (pdf_conf, year), text in pdf_text_by_cell.items():
            if pdf_conf != conf or not text:
                continue
            for candidate in candidates:
                role = name_in_pdf_with_role(candidate, text)
                if role:
                    pdf_rows.append({
                        "researcher_id": researcher_id,
                        "name": person["name"],
                        "conference": conf,
                        "year": year,
                        "role": role,
                        "matched_name": candidate,
                        "source": "ACM front-matter PDF",
                        "pdf_paths": pdf_paths_by_cell.get((conf, year), []),
                    })
                    break

    if (i + 1) % 100 == 0 or i + 1 == len(people):
        print(f"{i + 1}/{len(people)} researchers scanned in PDFs")

pdf_mentions = pd.DataFrame(pdf_rows)
if len(pdf_mentions):
    pdf_mentions = (
        pdf_mentions
        .drop_duplicates(["researcher_id", "conference", "year", "role"])
        .sort_values(["researcher_id", "conference", "year", "role"])
        .reset_index(drop=True)
    )
else:
    pdf_mentions = pd.DataFrame(columns=[
        "researcher_id", "name", "conference", "year", "role",
        "matched_name", "source", "pdf_paths",
    ])

print("PDF mentions:", pdf_mentions.shape)
if len(pdf_mentions):
    print(pdf_mentions["role"].value_counts().to_string())
    display(pdf_mentions.head())


100/952 researchers scanned in PDFs
200/952 researchers scanned in PDFs
300/952 researchers scanned in PDFs
400/952 researchers scanned in PDFs
500/952 researchers scanned in PDFs
600/952 researchers scanned in PDFs
700/952 researchers scanned in PDFs
800/952 researchers scanned in PDFs
900/952 researchers scanned in PDFs
952/952 researchers scanned in PDFs
PDF mentions: (11905, 8)
role
Other         4843
EPC           3572
PC            2789
Chair          471
Steering       213
Organizing      17


,researcher_id,name,conference,year,role,matched_name,source,pdf_paths
0,aaronbembenek,Aaron Bembenek,OOPSLA,2020,Other,Aaron Bembenek,ACM front-matter PDF,[assets/validation_pdfs/oopsla/oopsla_2020_acm...
1,aaronbembenek,Aaron Bembenek,OOPSLA,2024,Other,Aaron Bembenek,ACM front-matter PDF,[assets/validation_pdfs/oopsla/oopsla_2024_oop...
2,aaronbembenek,Aaron Bembenek,OOPSLA,2026,PC,Aaron Bembenek,ACM front-matter PDF,[assets/validation_pdfs/oopsla/oopsla_2026_oop...
3,aaronbembenek,Aaron Bembenek,PLDI,2025,PC,Aaron Bembenek,ACM front-matter PDF,[assets/validation_pdfs/pldi/pldi_2025_acm_fro...
4,aaronbembenek,Aaron Bembenek,POPL,2023,Other,Aaron Bembenek,ACM front-matter PDF,[assets/validation_pdfs/popl/popl_2023_acm_fro...


## 6 - Combine Evidence

In [12]:
PROFILE_PC_ROLES = {"PC Member", "PC Chair"}
PROFILE_BROAD_SERVICE_ROLES = {
    "PC Member", "PC Chair",
    "EPC Member", "EPC Chair",
    "General Chair", "Steering Chair", "Steering Committee",
    "Organizing", "Workshops Chair", "Publicity Chair", "Publications Chair",
}

PDF_PC_ROLES = {"PC"}
PDF_BROAD_SERVICE_ROLES = {"PC", "EPC", "Chair", "Steering", "Organizing"}


visible_evidence = pc_events.assign(
    role="Visible PC",
    source="current pc_members.parquet",
    evidence_detail=lambda d: d["conference"] + " " + d["year"].astype(str) + " (Visible PC)",
)
visible_evidence["is_pc_evidence"] = True
visible_evidence["is_broad_service_evidence"] = True

profile_evidence = profile_services.loc[
    profile_services["is_target_conference"],
    ["researcher_id", "conference", "year", "role", "source"]
].copy()
profile_evidence["is_pc_evidence"] = profile_evidence["role"].isin(PROFILE_PC_ROLES)
profile_evidence["is_broad_service_evidence"] = profile_evidence["role"].isin(PROFILE_BROAD_SERVICE_ROLES)
profile_conf = profile_evidence["conference"].astype("object").fillna("").astype(str)
profile_year = profile_evidence["year"].astype("object").fillna("").astype(str)
profile_role = profile_evidence["role"].astype("object").fillna("").astype(str)
profile_evidence["evidence_detail"] = (
    profile_conf + " " + profile_year + " (" + profile_role + ", Researchr profile)"
)

pdf_evidence = pdf_mentions[["researcher_id", "conference", "year", "role", "source"]].copy()
pdf_evidence["is_pc_evidence"] = pdf_evidence["role"].isin(PDF_PC_ROLES)
pdf_evidence["is_broad_service_evidence"] = pdf_evidence["role"].isin(PDF_BROAD_SERVICE_ROLES)
pdf_conf = pdf_evidence["conference"].astype("object").fillna("").astype(str)
pdf_year = pdf_evidence["year"].astype("object").fillna("").astype(str)
pdf_role = pdf_evidence["role"].astype("object").fillna("").astype(str)
pdf_evidence["evidence_detail"] = (
    pdf_conf + " " + pdf_year + " (" + pdf_role + ", ACM PDF)"
)

service_evidence = pd.concat(
    [visible_evidence, profile_evidence, pdf_evidence],
    ignore_index=True,
    sort=False,
)
service_evidence["year"] = service_evidence["year"].astype(int)
service_evidence = (
    service_evidence
    .drop_duplicates(["researcher_id", "conference", "year", "role", "source"])
    .sort_values(["researcher_id", "conference", "year", "source", "role"])
    .reset_index(drop=True)
)

print("all evidence rows:", service_evidence.shape)
print("PC evidence rows:", int(service_evidence["is_pc_evidence"].sum()))
print("broad service evidence rows:", int(service_evidence["is_broad_service_evidence"].sum()))
display(service_evidence.head())


all evidence rows: (19194, 8)
PC evidence rows: 7775
broad service evidence rows: 13494


,researcher_id,conference,year,role,source,evidence_detail,is_pc_evidence,is_broad_service_evidence
0,aaronbembenek,OOPSLA,2020,Other,ACM front-matter PDF,"OOPSLA 2020 (Other, ACM PDF)",False,False
1,aaronbembenek,OOPSLA,2024,Other,ACM front-matter PDF,"OOPSLA 2024 (Other, ACM PDF)",False,False
2,aaronbembenek,OOPSLA,2026,PC,ACM front-matter PDF,"OOPSLA 2026 (PC, ACM PDF)",True,True
3,aaronbembenek,OOPSLA,2026,PC Member,Researchr profile,"OOPSLA 2026 (PC Member, Researchr profile)",True,True
4,aaronbembenek,PLDI,2025,PC,ACM front-matter PDF,"PLDI 2025 (PC, ACM PDF)",True,True


In [13]:
def first_year(df, mask_col):
    sub = df.loc[df[mask_col]]
    if sub.empty:
        return pd.NA
    return int(sub["year"].min())


def detail_list(df, mask):
    vals = (
        df.loc[mask, "evidence_detail"]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    return vals


pair_rows = []
for _, pair in first_observed_conf.iterrows():
    researcher_id = pair["researcher_id"]
    conf = pair["conference"]
    cutoff = int(pair["first_observed_pc_year_conf"])
    ev = service_evidence.loc[
        (service_evidence["researcher_id"] == researcher_id)
        & (service_evidence["conference"] == conf)
    ].copy()

    pc_mask = ev["is_pc_evidence"]
    broad_mask = ev["is_broad_service_evidence"]
    prior_pc_mask = pc_mask & (ev["year"] < cutoff)
    prior_broad_mask = broad_mask & (ev["year"] < cutoff)

    pair_rows.append({
        "researcher_id": researcher_id,
        "conference": conf,
        "first_observed_pc_year_conf": cutoff,
        "source_verified_first_pc_year_conf": first_year(ev, "is_pc_evidence"),
        "source_verified_first_broad_service_year_conf": first_year(ev, "is_broad_service_evidence"),
        "had_prior_pc_before_observed_conf": bool(prior_pc_mask.any()),
        "had_prior_broad_service_before_observed_conf": bool(prior_broad_mask.any()),
        "prior_pc_evidence_count_conf": int(prior_pc_mask.sum()),
        "prior_broad_service_evidence_count_conf": int(prior_broad_mask.sum()),
        "prior_pc_evidence_conf": detail_list(ev, prior_pc_mask),
        "prior_broad_service_evidence_conf": detail_list(ev, prior_broad_mask),
    })

first_service_pairs = pd.DataFrame(pair_rows)

any_rows = []
for researcher_id, ev in service_evidence.groupby("researcher_id"):
    pc_ev = ev.loc[ev["is_pc_evidence"]]
    broad_ev = ev.loc[ev["is_broad_service_evidence"]]
    any_rows.append({
        "researcher_id": researcher_id,
        "source_verified_first_pc_year_any_target": int(pc_ev["year"].min()) if len(pc_ev) else pd.NA,
        "source_verified_first_broad_service_year_any_target": int(broad_ev["year"].min()) if len(broad_ev) else pd.NA,
    })

first_service_any = pd.DataFrame(any_rows)

first_service_pairs = (
    first_service_pairs
    .merge(first_observed_any, on="researcher_id", how="left")
    .merge(first_service_any, on="researcher_id", how="left")
    .merge(people[["researcher_id", "name"]], on="researcher_id", how="left")
)

first_service_pairs["is_true_first_pc_in_observed_year_conf"] = (
    first_service_pairs["source_verified_first_pc_year_conf"]
    == first_service_pairs["first_observed_pc_year_conf"]
)
first_service_pairs["is_true_first_broad_service_in_observed_year_conf"] = (
    first_service_pairs["source_verified_first_broad_service_year_conf"]
    == first_service_pairs["first_observed_pc_year_conf"]
)

first_service_pairs = first_service_pairs[[
    "researcher_id", "name", "conference",
    "first_observed_pc_year_conf",
    "source_verified_first_pc_year_conf",
    "source_verified_first_broad_service_year_conf",
    "had_prior_pc_before_observed_conf",
    "had_prior_broad_service_before_observed_conf",
    "prior_pc_evidence_count_conf",
    "prior_broad_service_evidence_count_conf",
    "prior_pc_evidence_conf",
    "prior_broad_service_evidence_conf",
    "is_true_first_pc_in_observed_year_conf",
    "is_true_first_broad_service_in_observed_year_conf",
    "first_observed_pc_year_any_target",
    "source_verified_first_pc_year_any_target",
    "source_verified_first_broad_service_year_any_target",
]]

display(first_service_pairs.head())


,researcher_id,name,conference,first_observed_pc_year_conf,source_verified_first_pc_year_conf,source_verified_first_broad_service_year_conf,had_prior_pc_before_observed_conf,had_prior_broad_service_before_observed_conf,prior_pc_evidence_count_conf,prior_broad_service_evidence_count_conf,prior_pc_evidence_conf,prior_broad_service_evidence_conf,is_true_first_pc_in_observed_year_conf,is_true_first_broad_service_in_observed_year_conf,first_observed_pc_year_any_target,source_verified_first_pc_year_any_target,source_verified_first_broad_service_year_any_target
0,aaronbembenek,Aaron Bembenek,PLDI,2025,2025,2025,False,False,0,0,[],[],True,True,2025,2025,2025
1,aaronstump,Aaron Stump,POPL,2019,2019,2005,False,True,0,6,[],"[POPL 2005 (EPC, ACM PDF), POPL 2006 (EPC, ACM...",True,False,2019,2019,2005
2,abhinavverma1,Abhinav Verma,PLDI,2023,2023,2023,False,False,0,0,[],[],True,True,2023,2023,2023
3,adamchlipala,Adam Chlipala,ICFP,2017,2011,2008,True,True,1,8,"[ICFP 2011 (PC, ACM PDF)]","[ICFP 2008 (EPC, ACM PDF), ICFP 2009 (EPC, ACM...",False,False,2017,2011,2008
4,adamchlipala,Adam Chlipala,PLDI,2018,2018,2008,False,True,0,5,[],"[PLDI 2008 (EPC, ACM PDF), PLDI 2011 (EPC, ACM...",True,False,2017,2011,2008


## 7 - Save Outputs

In [14]:
pc_members_with_first_service = (
    pc_members
    .merge(
        first_service_pairs.drop(columns=["name"]),
        on=["researcher_id", "conference"],
        how="left",
    )
    .sort_values(["conference", "year", "name"])
    .reset_index(drop=True)
)

profiles.to_parquet(intermediate_dir / "researchr_profile_fetch_log.parquet", index=False)
profile_services.to_parquet(intermediate_dir / "researchr_profile_services.parquet", index=False)
pdf_mentions.to_parquet(intermediate_dir / "acm_pdf_service_mentions.parquet", index=False)
service_evidence.to_parquet(intermediate_dir / "pc_service_evidence.parquet", index=False)
first_service_pairs.to_parquet(prepared_dir / "pc_first_service_evidence.parquet", index=False)
pc_members_with_first_service.to_parquet(intermediate_dir / "pc_members_with_first_service.parquet", index=False)

summary = (
    first_service_pairs
    .groupby("conference")
    .agg(
        researcher_conference_pairs=("researcher_id", "size"),
        pairs_with_prior_pc=("had_prior_pc_before_observed_conf", "sum"),
        pairs_with_prior_broad_service=("had_prior_broad_service_before_observed_conf", "sum"),
        true_first_pc_pairs=("is_true_first_pc_in_observed_year_conf", "sum"),
        true_first_broad_service_pairs=("is_true_first_broad_service_in_observed_year_conf", "sum"),
    )
    .reset_index()
)

summary["share_with_prior_pc"] = (
    summary["pairs_with_prior_pc"] / summary["researcher_conference_pairs"]
).round(3)
summary["share_with_prior_broad_service"] = (
    summary["pairs_with_prior_broad_service"] / summary["researcher_conference_pairs"]
).round(3)

summary.to_csv(summary_tables_dir / "pc_first_service_summary.csv", index=False)

profile_status = (
    profiles["profile_fetch_status"]
    .value_counts(dropna=False)
    .rename_axis("profile_fetch_status")
    .reset_index(name="count")
)
profile_status.to_csv(dependency_tables_dir / "researchr_profile_fetch_status.csv", index=False)

print("Wrote:")
for path in [
    intermediate_dir / "researchr_profile_fetch_log.parquet",
    intermediate_dir / "researchr_profile_services.parquet",
    intermediate_dir / "acm_pdf_service_mentions.parquet",
    intermediate_dir / "pc_service_evidence.parquet",
    prepared_dir / "pc_first_service_evidence.parquet",
    intermediate_dir / "pc_members_with_first_service.parquet",
    summary_tables_dir / "pc_first_service_summary.csv",
    dependency_tables_dir / "researchr_profile_fetch_status.csv",
]:
    print(" ", path.relative_to(repo))

display(summary)


Wrote:
  step_1_data/intermediate/researchr_profile_fetch_log.parquet
  step_1_data/intermediate/researchr_profile_services.parquet
  step_1_data/intermediate/acm_pdf_service_mentions.parquet
  step_1_data/intermediate/pc_service_evidence.parquet
  step_1_data/prepared/pc_first_service_evidence.parquet
  step_1_data/intermediate/pc_members_with_first_service.parquet
  step_1_artifacts/summary_tables/pc_first_service_summary.csv
  step_1_artifacts/dependency_tables/researchr_profile_fetch_status.csv


,conference,researcher_conference_pairs,pairs_with_prior_pc,pairs_with_prior_broad_service,true_first_pc_pairs,true_first_broad_service_pairs,share_with_prior_pc,share_with_prior_broad_service
0,ICFP,250,77,184,173,66,0.308,0.736
1,OOPSLA,382,81,207,301,175,0.212,0.542
2,PLDI,480,123,309,357,171,0.256,0.644
3,POPL,384,127,292,257,92,0.331,0.760


## Output Meaning

`pc_first_service_evidence.parquet` is the main Step 1 validation table.
It has one row per researcher and conference pair from `pc_members.parquet`.

Important columns:

- `first_observed_pc_year_conf`: first PC year in the current PC service data.
- `source_verified_first_pc_year_conf`: earliest main PC year found from
  visible PC data, Researchr profiles, or ACM PDFs for the same conference.
- `source_verified_first_broad_service_year_conf`: earliest broader review
  service year found for the same conference.
- `had_prior_pc_before_observed_conf`: whether there is main PC evidence
  before the first observed PC year.
- `had_prior_broad_service_before_observed_conf`: whether there is broader
  review service evidence before the first observed PC year.

The row level `pc_members_with_first_service.parquet` attaches these fields
back to every PC service row.


## File To Use Later

I keep `pc_members.parquet` as the base PC data.
When I need the first service validation fields, I should join in
`pc_first_service_evidence.parquet`.

```python
pc = pd.read_parquet(prepared_dir / "pc_members.parquet")
first = pd.read_parquet(prepared_dir / "pc_first_service_evidence.parquet")

pc_with_first_service = pc.merge(
    first.drop(columns=["name"]),
    on=["researcher_id", "conference"],
    how="left",
)
```